# EV purchase prediction: reproducible baseline

This notebook loads the **Playground Series S6E9** data, builds extra features, trains five-fold LightGBM and CatBoost models, chooses a blend using out-of-fold ROC AUC, and writes `submission.csv`.

Each run is one named **experiment** (`CFG.EXPERIMENT`). Its scores are appended to `experiments.csv`, its predictions are saved in `preds/`, and its submission is also saved as `submission_<experiment>.csv`. All experiments share the fold assignment stored in `folds_seed42.csv`, so their scores are directly comparable.

Run the cells from top to bottom. Keep `train.csv`, `test.csv`, and `sample_submission.csv` beside this notebook for local execution. On Kaggle, the notebook can also find the files under `/kaggle/input`.

Every nontrivial code statement has a nearby comment explaining its role. The validation score estimates model quality; `submission.csv` contains test-set probabilities in the competition's requested row order.

## 1. Imports and configuration

All settings live in one place. `DEBUG` uses fewer rows and trees for a quick check; leave it `False` for the full submission.

In [1]:
import gc  # Release temporary fold objects after each fit.
import random  # Seed Python's random number generator.
import time  # Measure cross-validation runtime.
from pathlib import Path  # Work with local and Kaggle paths.

import numpy as np  # Store predictions and do numerical calculations.
import pandas as pd  # Load CSV files and engineer tabular features.
import torch  # Train the neural network. Import before LightGBM/XGBoost: the other order crashes PyTorch on macOS.
import torch.nn as nn  # Neural network layers.
import lightgbm as lgb  # Train the first gradient-boosted model.
import xgboost as xgb  # Train the third boosted model.
from sklearn.preprocessing import QuantileTransformer  # Scale numbers to a normal shape for the network.
import plotly.graph_objects as go  # Build the diagnostic charts.
import plotly.io as pio  # Configure a shared chart theme.
from plotly.subplots import make_subplots  # Arrange category charts in a grid.
from catboost import CatBoostClassifier, Pool  # Train the second boosted model.
from sklearn.metrics import log_loss, roc_auc_score  # Evaluate predictions.
from sklearn.model_selection import StratifiedKFold  # Preserve class balance in folds.


class CFG:
    EXPERIMENT = "exp10_lgbm_params"  # Name for experiments.csv, preds/, and the submission copy.
    BASELINE = "exp09_wide_encoding"  # Current best experiment; deltas are measured against it.
    DEBUG = False  # Set True to check the workflow quickly on 20,000 rows.
    FAST = False  # Set True to screen a feature idea with LightGBM only at a higher learning rate.
    CLIP_FLAGS = True  # Exp01: flag rows sitting exactly on the income and commute floors.
    TARGET_ENCODING = True  # Exp02: add in-fold target means for the columns and pairs below.
    COUNT_ENCODING = True  # Exp03: add how often each TE_COLS value appears in train + test (label-free).
    TE_COLS = [  # Numeric columns with few distinct values behave like categories.
        "Age", "Annual_Income_USD", "Daily_Commute_km",
        "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",
    ]
    TE_WIDE = True  # Exp09: also encode digits, coarse bins, and more low-cardinality columns.
    TE_WIDE_EXTRA = [  # Added to TE_COLS (and to count encoding) when TE_WIDE is on.
        "Income_Digit_1", "Income_Digit_10", "Income_Digit_100", "Income_Digit_1000",
        "Commute_Digit_Tenths", "Commute_Digit_Units", "Age_Digit_Units",
        "Income_Bin100", "Income_Bin1000", "Commute_Int",
        "Number_of_Cars_Owned", "Environmental_Concern_Level",
    ]
    TE_SMOOTHS = (10, 100)  # Exp09: each key is encoded at every smoothing strength (rows of prior weight).
    RECIPE_FEATURES = False  # Screened +0.00002 alone and -0.00005 on top of wide encoding: trees already learn the recipe.
    CB_NUM_COLS = [  # Numeric columns CatBoost also sees as categories (Exp04).
        "Age", "Annual_Income_USD", "Daily_Commute_km",
        "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",
    ]
    TE_PAIRS = []  # Pair interactions, e.g. ("Age", "City_Type"); two pairs screened at +0.00001, so unused.
    DIGIT_FEATURES = True  # Exp06: individual digits of income, commute, and age (synthetic-data artifacts).
    ORIG_FEATURES = False  # Exp05 (screened +0.00000, unused): each value's purchase rate and frequency in the original dataset.
    ORIG_FILE = "EV_Adoption_and_Range_Anxiety_Dataset.csv"  # Kaggle dataset: itzzomkar/ev-adoption-behavior-and-range-anxiety.
    ORIG_SMOOTH = 0  # Prior rows added to each original-data rate (0 = raw rate).
    CB_NUM_AS_CAT = True  # Exp04: CatBoost also sees TE_COLS as categories (integer copies named CAT_<column>).
    CB_CTR_COMPLEXITY = 1  # CatBoost category combinations; 2 screened no better on fold 1 (0.94467 vs 0.94462).
    RUN_CATBOOST = True  # Set False to train only LightGBM.
    RUN_XGBOOST = True  # Exp07: add XGBoost as a third model.
    RUN_NN = True  # Exp08: add a neural network (MLP with periodic number embeddings) as a fourth model.
    NN_EPOCHS = 12  # Passes over the training rows; the best epoch on validation is kept.
    NN_HIDDEN = (256, 128, 64)  # Hidden layer widths.
    NN_DROPOUT = 0.15  # Share of hidden units dropped during training.
    NN_LR = 1e-3  # Peak learning rate of the one-cycle schedule.
    NN_BATCH = 1024  # Rows per gradient step.
    PLR_K = 24  # Sine/cosine frequencies per numeric feature.
    PLR_DIM = 12  # Output size of each numeric feature's embedding.
    PLR_SIGMA = 0.1  # Spread of the initial frequencies.
    SEED = 42  # Use the same random seed in splitting and training.
    N_FOLDS = 5  # Train and validate on five fixed partitions.
    METRIC = "auc"  # Change to "logloss" if the competition metric changes.
    TARGET = "Will_Buy_EV"  # Name of the label in train.csv.
    ID = "id"  # Key used to restore the submission's row order.
    EARLY_STOP = 200  # Stop when validation quality stops improving.
    LGB_ITERS = 5000  # Maximum LightGBM trees before early stopping.
    LGB_LR = 0.03  # LightGBM learning rate.
    LGB_PARAMS = dict(  # Exp10: shallow trees that each see 30% of the features (from top public notebooks); +0.00022 in screening.
        num_leaves=32, max_depth=5, min_child_samples=10, subsample=0.81,
        colsample_bytree=0.30, reg_alpha=0.07, reg_lambda=2.0, max_bin=255,
    )
    CB_ITERS = 3000  # Maximum CatBoost trees before early stopping.
    XGB_ITERS = 5000  # Maximum XGBoost trees before early stopping.
    CAT_COLS = [  # Treat these columns as categories rather than numbers.
        "Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible",
        "Subsidy_Available", "Range_Anxiety_Level", "Environmental_Concern_Level",
    ]


if CFG.TE_WIDE:  # Encode the wider set of keys.
    CFG.TE_COLS = CFG.TE_COLS + [name for name in CFG.TE_WIDE_EXTRA if name not in CFG.TE_COLS]

if CFG.DEBUG:  # Shorten both models when testing code changes.
    CFG.LGB_ITERS, CFG.CB_ITERS, CFG.XGB_ITERS, CFG.EARLY_STOP, CFG.NN_EPOCHS = 300, 300, 300, 50, 2
if CFG.FAST:  # FAST scores are only comparable with other FAST runs.
    CFG.RUN_CATBOOST, CFG.RUN_XGBOOST, CFG.RUN_NN, CFG.LGB_LR = False, False, False, 0.05

random.seed(CFG.SEED)  # Make Python-side random choices repeatable.
np.random.seed(CFG.SEED)  # Make NumPy-side random choices repeatable.


def score(y_true, probability):
    """Return the selected competition metric for probability predictions."""
    if CFG.METRIC == "auc":  # Higher AUC means better ranking.
        return roc_auc_score(y_true, probability)
    return log_loss(y_true, probability)  # Lower log loss means better calibration.


def find_data_dir():
    """Prefer nearby CSVs, then look for the Kaggle competition mount."""
    required = ("train.csv", "test.csv", "sample_submission.csv")  # Input file names.
    local = Path.cwd()  # The notebook's working directory.
    if all((local / name).is_file() for name in required):  # Local run.
        return local
    kaggle = Path("/kaggle/input")  # Kaggle's read-only input mount.
    if kaggle.exists():  # Search only when the mount is available.
        matches = sorted(kaggle.rglob("train.csv"))  # Find candidate datasets.
        matches.sort(key=lambda path: "playground-series-s6e9" not in str(path))  # Prefer this competition.
        for path in matches:  # Check that the companion files are present.
            if all((path.parent / name).is_file() for name in required):
                return path.parent
    raise FileNotFoundError("Place train.csv, test.csv, and sample_submission.csv beside this notebook.")


DATA_DIR = find_data_dir()  # Resolve the input location once.
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()  # Save the submission where writes are allowed.
print(f"Experiment: {CFG.EXPERIMENT} | Input: {DATA_DIR} | Output: {OUT_DIR} | Debug: {CFG.DEBUG} | Fast: {CFG.FAST}")  # Show the active paths and mode.

Experiment: exp10_lgbm_params | Input: /Users/rodan/Documents/kaggle-september | Output: /Users/rodan/Documents/kaggle-september | Debug: False | Fast: False


## 2. Load and check the data

These checks fail early if the files are from a different competition or their rows cannot form a valid submission.

In [2]:
train = pd.read_csv(DATA_DIR / "train.csv")  # Read labelled training rows.
test = pd.read_csv(DATA_DIR / "test.csv")  # Read rows that need predictions.
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")  # Read the required output layout.

assert CFG.TARGET in train.columns, f"Missing target: {CFG.TARGET}"  # Confirm the label exists.
assert set(train.columns) - {CFG.TARGET} == set(test.columns), "Train/test columns differ."  # Confirm matching inputs.
assert CFG.ID in train and CFG.ID in test and CFG.ID in sample, "An id column is missing."  # Confirm join keys.
assert len(sample) == len(test), "Submission and test row counts differ."  # Confirm expected output size.
assert sample[CFG.ID].is_unique and test[CFG.ID].is_unique, "Test IDs must be unique."  # Prevent duplicate joins.
assert set(sample[CFG.ID]) == set(test[CFG.ID]), "Submission IDs differ from test IDs."  # Prevent missing predictions.

if not pd.api.types.is_numeric_dtype(train[CFG.TARGET]):  # Handle Yes/No labels if supplied as text.
    train[CFG.TARGET] = train[CFG.TARGET].astype(str).str.strip().str.lower().map({"no": 0, "yes": 1})
assert train[CFG.TARGET].isin([0, 1]).all(), "The target must contain only 0 and 1."  # Reject invalid labels.
train[CFG.TARGET] = train[CFG.TARGET].astype("int8")  # Store the binary label compactly.

if CFG.DEBUG:  # Use a reproducible subset for a quick workflow check.
    train = train.sample(n=min(20_000, len(train)), random_state=CFG.SEED).reset_index(drop=True)

for frame in (train, test):  # Reduce memory use for the large input tables.
    for column in frame.select_dtypes(include="float64").columns:  # Float32 is sufficient for these features.
        frame[column] = frame[column].astype("float32")
    for column in frame.select_dtypes(include="int64").columns:  # Downcast integer features except IDs.
        if column != CFG.ID:
            frame[column] = pd.to_numeric(frame[column], downcast="integer")

print("train:", train.shape, "test:", test.shape, "sample:", sample.shape)  # Check data sizes.
print("missing cells:", int(train.isna().sum().sum()), int(test.isna().sum().sum()))  # Check input completeness.
print("positive rate:", round(train[CFG.TARGET].mean(), 4))  # Check class balance.

train: (668665, 15) test: (286571, 14) sample: (286571, 2)
missing cells: 0 0
positive rate: 0.1746


## 3. Feature engineering

The same transformations are applied to train and test. Category vocabularies come from both input tables; this changes feature encoding only and does not use test labels.

In [3]:
ANXIETY_ORDER = {"low": 0, "medium": 1, "high": 2}  # Numeric ordering for text anxiety levels.


def make_features(frame):
    """Add the derived features used by both models."""
    result = frame.copy()  # Avoid mutating the raw input table.
    columns = set(result.columns)  # Make optional-column checks quick.
    if {"Charging_Stations_Near_Home", "Charging_Stations_Near_Work"} <= columns:  # Both counts available.
        result["Total_Stations"] = result["Charging_Stations_Near_Home"] + result["Charging_Stations_Near_Work"]  # Total charging access.
    if {"Annual_Income_USD", "Number_of_Cars_Owned"} <= columns:  # Both inputs available.
        result["Income_per_Car"] = result["Annual_Income_USD"] / (result["Number_of_Cars_Owned"] + 1)  # Avoid division by zero.
    if "Range_Anxiety_Level" in columns:  # Create an ordered anxiety feature.
        anxiety = result["Range_Anxiety_Level"]  # Read the source category.
        if not pd.api.types.is_numeric_dtype(anxiety):  # Convert Low/Medium/High text when needed.
            anxiety = anxiety.astype(str).str.strip().str.lower().map(ANXIETY_ORDER)
        result["Anxiety_Ord"] = anxiety.astype("float32")  # Store the ordered value.
        if "Daily_Commute_km" in columns:  # Measure commute exposure to range anxiety.
            result["Commute_x_Anxiety"] = result["Daily_Commute_km"] * (result["Anxiety_Ord"] + 1)
    if "Environmental_Concern_Level" in columns and pd.api.types.is_numeric_dtype(result["Environmental_Concern_Level"]):  # Keep its numeric interpretation.
        result["Env_Concern_Num"] = result["Environmental_Concern_Level"].astype("float32")
    if CFG.CLIP_FLAGS:  # Many rows sit exactly on a floor value, likely a clipping artifact of the data generator.
        if "Annual_Income_USD" in columns:  # About 9% of rows report exactly 30,000.
            result["Income_is_Floor"] = (result["Annual_Income_USD"] == 30000).astype("int8")
        if "Daily_Commute_km" in columns:  # About 22% of rows report exactly 5.0 km.
            result["Commute_is_Floor"] = (result["Daily_Commute_km"] == 5.0).astype("int8")
    if CFG.TE_WIDE:  # Coarse keys: neighbouring values share a bucket, so each bucket has more rows.
        if "Annual_Income_USD" in columns:
            result["Income_Bin100"] = np.floor(result["Annual_Income_USD"] / 100).astype("int32")
            result["Income_Bin1000"] = np.floor(result["Annual_Income_USD"] / 1000).astype("int32")
        if "Daily_Commute_km" in columns:
            result["Commute_Int"] = np.floor(result["Daily_Commute_km"]).astype("int32")
    if CFG.RECIPE_FEATURES:  # Scores reverse-engineered from the original data generator (Chris Deotte's notebook).
        anxiety_penalty = result["Anxiety_Ord"].map({0.0: 0.0, 1.0: 1.0, 2.0: 3.0}).astype("float32")  # Medium -1, High -3.
        subsidy = (result["Subsidy_Available"].astype(str) == "Yes").astype("float32")
        home_charging = (result["Home_Charging_Possible"].astype(str) == "Yes").astype("float32")
        result["Buy_Score"] = (1.2 * result["Annual_Income_USD"] / 100_000 + 0.6 * result["Environmental_Concern_Level"]
                               + 2.0 * subsidy - anxiety_penalty).astype("float32")  # Buys when this exceeds about 5.5.
        result["Worry_Score"] = (result["Daily_Commute_km"] - 5 * result["Charging_Stations_Near_Home"]
                                 - 5 * result["Charging_Stations_Near_Work"] - 150 * home_charging).astype("float32")
        result["Is_Millionaire_Cliff"] = (result["Annual_Income_USD"] >= 170537).astype("int8")  # 100% bought in train.
        result["Is_Dead_Zone"] = ((result["Annual_Income_USD"] >= 38000) & (result["Annual_Income_USD"] <= 42000)).astype("int8")  # ~0% bought.
        result["Is_Env_Hater"] = (result["Environmental_Concern_Level"] == 1).astype("int8")  # 0.6% bought.
    if CFG.DIGIT_FEATURES:  # The data generator may treat individual digits or round numbers specially.
        if "Annual_Income_USD" in columns:
            income = result["Annual_Income_USD"].round().astype("int64")  # Incomes are whole dollars.
            for power in range(4):  # Ones, tens, hundreds, and thousands digits.
                result[f"Income_Digit_{10 ** power}"] = ((income // 10 ** power) % 10).astype("int8")
            trailing = np.zeros(len(income), dtype="int8")  # Count trailing zeros: 60000 -> 4.
            for power in range(1, 6):
                trailing += (income % 10 ** power == 0).to_numpy()
            result["Income_Trailing_Zeros"] = trailing
        if "Daily_Commute_km" in columns:
            tenths = (result["Daily_Commute_km"].astype("float64") * 10).round().astype("int64")  # One decimal place.
            result["Commute_Digit_Tenths"] = (tenths % 10).astype("int8")
            result["Commute_Digit_Units"] = ((tenths // 10) % 10).astype("int8")
        if "Age" in columns:
            result["Age_Digit_Units"] = (result["Age"] % 10).astype("int8")
    return result  # Return the expanded table.


def find_original():
    """Locate the original dataset beside the competition files or under /kaggle/input."""
    for folder in (DATA_DIR, Path.cwd()):  # Local runs keep it beside train.csv.
        if (folder / CFG.ORIG_FILE).is_file():
            return folder / CFG.ORIG_FILE
    matches = sorted(Path("/kaggle/input").rglob(CFG.ORIG_FILE)) if Path("/kaggle/input").exists() else []  # Attached Kaggle dataset.
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Add {CFG.ORIG_FILE} (Kaggle: itzzomkar/ev-adoption-behavior-and-range-anxiety).")


def value_key(series):
    """Comparable key per value: numbers in tenths as integers (safe across float32/float64), text stripped."""
    if pd.api.types.is_numeric_dtype(series):
        return (series.astype("float64") * 10).round().astype("Int64")
    return series.astype(str).str.strip()


def add_original_features(frames, original):
    """Add each value's purchase rate (ORIG_TE_) and row count (ORIG_CNT_) in the original dataset.

    Only the original dataset's labels are used, never train.csv's, so this is computed once outside the folds.
    """
    y_original = original[CFG.TARGET].astype(str).str.strip().str.lower().eq("yes").astype(float)  # Yes/No to 1/0.
    prior = y_original.mean()  # Overall purchase rate in the original data.
    shared = [column for column in original.columns if column != CFG.TARGET and all(column in frame for frame in frames)]
    for name in shared:  # Raw input columns present everywhere; never the target itself.
        stats = (pd.DataFrame({"key": value_key(original[name]), "y": y_original}).dropna()
                 .groupby("key")["y"].agg(["sum", "count"]))  # Buyers and rows per value; missing values skipped.
        rate = (stats["sum"] + prior * CFG.ORIG_SMOOTH) / (stats["count"] + CFG.ORIG_SMOOTH)
        for frame in frames:
            key = value_key(frame[name])
            frame["ORIG_TE_" + name] = key.map(rate).astype("float32")  # NaN when the value never appears in the original.
            frame["ORIG_CNT_" + name] = key.map(stats["count"]).fillna(0).astype("float32")


train = make_features(train)  # Engineer the training features.
test = make_features(test)  # Engineer the identical test features.
if CFG.ORIG_FEATURES:  # Look up every value in the dataset the competition data was generated from.
    original = pd.read_csv(find_original())
    add_original_features([train, test], original.drop(columns=["Buyer_ID"], errors="ignore"))
    print(f"Original dataset: {original.shape}; added {sum(c.startswith('ORIG_') for c in test.columns)} features")
if "Anxiety_Ord" in train and train["Anxiety_Ord"].isna().all():  # Catch unexpected anxiety labels.
    raise ValueError("Range_Anxiety_Level could not be mapped to low/medium/high.")

CFG.CAT_COLS = [name for name in CFG.CAT_COLS if name in test.columns]  # Keep only available categories.
FEATURES = [name for name in test.columns if name != CFG.ID]  # Exclude the ID from modeling.
NUM_FEATS = [name for name in FEATURES if name not in CFG.CAT_COLS]  # Record numerical feature names.

for name in CFG.CAT_COLS:  # Give train and test exactly the same category codes.
    vocabulary = sorted(set(train[name].astype(str)) | set(test[name].astype(str)))  # Include both tables' values.
    train[name] = pd.Categorical(train[name].astype(str), categories=vocabulary)  # Encode train categories.
    test[name] = pd.Categorical(test[name].astype(str), categories=vocabulary)  # Encode test categories.

def encoding_keys():
    """Give every encoded column or pair one integer code per distinct value, shared by train and test."""
    both = pd.concat([train, test], ignore_index=True)  # Same codes in both tables; no labels involved.
    keys = {}
    for group in [(name,) for name in CFG.TE_COLS] + [tuple(pair) for pair in CFG.TE_PAIRS]:
        text = both[group[0]].astype(str)  # Start the combined key with the first column.
        for name in group[1:]:  # Append the pair's second column.
            text = text + "|" + both[name].astype(str)
        keys["_x_".join(group)] = pd.factorize(text)[0].astype("int32")
    return keys


ENCODING_KEYS = encoding_keys() if CFG.TARGET_ENCODING or CFG.COUNT_ENCODING else {}  # Group code per row.
TRAIN_KEYS = {name: codes[:len(train)] for name, codes in ENCODING_KEYS.items()}  # Train rows come first.
TEST_KEYS = {name: codes[len(train):] for name, codes in ENCODING_KEYS.items()}  # Then test rows.
N_CODES = {name: int(codes.max()) + 1 for name, codes in ENCODING_KEYS.items()}  # Distinct values per key.

if CFG.COUNT_ENCODING:  # Frequency of each value; rare values often behave differently.
    for name, codes in ENCODING_KEYS.items():
        counts = np.bincount(codes)  # Occurrences of each group code across train + test.
        train["CNT_" + name] = counts[TRAIN_KEYS[name]].astype("int32")
        test["CNT_" + name] = counts[TEST_KEYS[name]].astype("int32")
    FEATURES = [name for name in test.columns if name != CFG.ID]  # Include the count columns.
    NUM_FEATS = [name for name in FEATURES if name not in CFG.CAT_COLS]

print(f"{len(FEATURES)} features: {len(CFG.CAT_COLS)} categorical, {len(NUM_FEATS)} numeric")  # Summarize model inputs.
print(FEATURES)  # Make the feature list easy to audit.

48 features: 7 categorical, 41 numeric
['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level', 'Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level', 'Total_Stations', 'Income_per_Car', 'Anxiety_Ord', 'Commute_x_Anxiety', 'Env_Concern_Num', 'Income_is_Floor', 'Commute_is_Floor', 'Income_Bin100', 'Income_Bin1000', 'Commute_Int', 'Income_Digit_1', 'Income_Digit_10', 'Income_Digit_100', 'Income_Digit_1000', 'Income_Trailing_Zeros', 'Commute_Digit_Tenths', 'Commute_Digit_Units', 'Age_Digit_Units', 'CNT_Age', 'CNT_Annual_Income_USD', 'CNT_Daily_Commute_km', 'CNT_Charging_Stations_Near_Home', 'CNT_Charging_Stations_Near_Work', 'CNT_Income_Digit_1', 'CNT_Income_Digit_10', 'CNT_Income_Digit_100', 'CNT_Income_Digit_1000', 'CNT_Commute_Digit_Tenths', 'CNT_Commute_Digit_Units', 'CNT_Age_Digit_Units', 'CNT_Income_Bin100', 'CNT_Incom

## 4. Five-fold model training

The same stratified folds evaluate both models. Each training row receives one out-of-fold prediction, and each test row receives the average of five fold predictions.

In [4]:
y = train[CFG.TARGET].to_numpy()  # Keep labels as a compact NumPy array.
fold_path = OUT_DIR / f"folds_seed{CFG.SEED}.csv"  # Saved fold IDs keep every experiment on identical splits.
if fold_path.is_file() and not CFG.DEBUG:  # Reuse the saved assignment.
    saved_folds = pd.read_csv(fold_path)
    assert saved_folds[CFG.ID].equals(train[CFG.ID].astype(saved_folds[CFG.ID].dtype)), "Saved folds do not match train.csv."
    fold_id = saved_folds["fold"].to_numpy().astype("int8")
    print(f"Loaded folds from {fold_path.name}")
else:  # Build the stratified assignment once.
    fold_id = np.full(len(train), -1, dtype="int8")  # Reserve one fold number per training row.
    splitter = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)  # Fix balanced partitions.
    for number, (_, valid_idx) in enumerate(splitter.split(train, y)):  # Assign each validation partition.
        fold_id[valid_idx] = number
    if not CFG.DEBUG:  # A 20k-row debug sample must not overwrite the full assignment.
        pd.DataFrame({CFG.ID: train[CFG.ID], "fold": fold_id}).to_csv(fold_path, index=False)
assert (fold_id >= 0).all(), "Every training row needs a fold."  # Check fold coverage.


def smoothed_mean(fit_codes, fit_y, apply_codes, n_codes, prior, smooth):
    """Purchase rate per group, shrunk toward the prior for rare groups."""
    total = np.bincount(fit_codes, weights=fit_y, minlength=n_codes)  # Buyers per group.
    count = np.bincount(fit_codes, minlength=n_codes)  # Rows per group.
    return ((total + prior * smooth) / (count + smooth))[apply_codes].astype("float32")


def add_target_encoding(train_idx, valid_idx, x_train, x_valid, x_test):
    """Add TE_ columns using only this fold's training labels.

    Validation and test rows get means from all training rows. Training rows get means from an
    inner 5-fold split, so no training row's encoding includes its own label.
    """
    y_fit = y[train_idx]  # Labels the encoder may use.
    prior = y_fit.mean()  # Fallback rate for unseen or rare values.
    inner = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=CFG.SEED).split(train_idx, y_fit))
    new_train, new_valid, new_test = {}, {}, {}
    for name, codes in TRAIN_KEYS.items():
        fit_codes = codes[train_idx]  # Group codes of the training rows.
        for smooth in CFG.TE_SMOOTHS:  # A light and a heavy smoothing: fine detail and a stable trend.
            label = f"TE{smooth}_{name}"
            inner_values = np.zeros(len(train_idx), dtype="float32")
            for fit_part, apply_part in inner:  # Out-of-fold encoding inside the training rows.
                inner_values[apply_part] = smoothed_mean(fit_codes[fit_part], y_fit[fit_part], fit_codes[apply_part], N_CODES[name], prior, smooth)
            new_train[label] = inner_values
            new_valid[label] = smoothed_mean(fit_codes, y_fit, codes[valid_idx], N_CODES[name], prior, smooth)
            new_test[label] = smoothed_mean(fit_codes, y_fit, TEST_KEYS[name], N_CODES[name], prior, smooth)
    return x_train.assign(**new_train), x_valid.assign(**new_valid), x_test.assign(**new_test)


def fit_lightgbm(x_train, y_train, x_valid, y_valid, x_test):
    """Fit one LightGBM fold and return validation and test probabilities."""
    model = lgb.LGBMClassifier(  # Use the original baseline's tree settings.
        n_estimators=CFG.LGB_ITERS, learning_rate=CFG.LGB_LR, subsample_freq=1,
        random_state=CFG.SEED, verbose=-1, feature_pre_filter=False, **CFG.LGB_PARAMS,
    )
    evaluation = "auc" if CFG.METRIC == "auc" else "binary_logloss"  # Match the chosen score.
    model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)], eval_metric=evaluation,
              callbacks=[lgb.early_stopping(CFG.EARLY_STOP, verbose=False)])  # Stop using validation data.
    fit_lightgbm.last_model = model  # Reuse the last fold for feature importance.
    valid_probability = model.predict_proba(x_valid, num_iteration=model.best_iteration_)[:, 1]  # Validation scores.
    test_probability = model.predict_proba(x_test, num_iteration=model.best_iteration_)[:, 1]  # Test scores.
    return valid_probability, test_probability  # Hand both arrays to the CV loop.


def category_codes(frame):
    """Convert shared pandas categories to integer values for CatBoost."""
    result = frame.assign(**{name: frame[name].cat.codes.astype("int32") for name in CFG.CAT_COLS})  # Keep other columns.
    if CFG.CB_NUM_AS_CAT:  # Integer copies (x10 keeps one decimal) so each numeric value can act as its own category.
        result = result.assign(**{f"CAT_{name}": (frame[name] * 10).round().astype("int64") for name in CFG.CB_NUM_COLS})
    return result


CB_CAT_FEATURES = CFG.CAT_COLS + ([f"CAT_{name}" for name in CFG.CB_NUM_COLS] if CFG.CB_NUM_AS_CAT else [])  # CatBoost categories.


def fit_catboost(x_train, y_train, x_valid, y_valid, x_test):
    """Fit one CPU CatBoost fold and return validation and test probabilities."""
    model = CatBoostClassifier(  # Match the original baseline's CatBoost settings.
        iterations=CFG.CB_ITERS, learning_rate=0.08, depth=6, l2_leaf_reg=3,
        max_ctr_complexity=CFG.CB_CTR_COMPLEXITY, border_count=128, loss_function="Logloss",
        eval_metric="AUC" if CFG.METRIC == "auc" else "Logloss",
        od_type="Iter", od_wait=CFG.EARLY_STOP, use_best_model=True,
        random_seed=CFG.SEED, verbose=False, thread_count=-1,
        allow_writing_files=False,  # Avoid creating CatBoost log files on each run.
    )
    train_pool = Pool(category_codes(x_train), y_train, cat_features=CB_CAT_FEATURES)  # Label training categories.
    valid_pool = Pool(category_codes(x_valid), y_valid, cat_features=CB_CAT_FEATURES)  # Label validation categories.
    model.fit(train_pool, eval_set=valid_pool)  # Fit and choose the best iteration.
    valid_probability = model.predict_proba(category_codes(x_valid))[:, 1]  # Validation scores.
    test_probability = model.predict_proba(category_codes(x_test))[:, 1]  # Test scores.
    return valid_probability, test_probability  # Hand both arrays to the CV loop.


def fit_xgboost(x_train, y_train, x_valid, y_valid, x_test):
    """Fit one XGBoost fold and return validation and test probabilities."""
    model = xgb.XGBClassifier(  # Depth-wise trees and native categories: different mistakes from LightGBM and CatBoost.
        n_estimators=CFG.XGB_ITERS, learning_rate=0.03, max_depth=7, min_child_weight=5,
        subsample=0.8, colsample_bytree=0.7, reg_lambda=1.0, tree_method="hist",
        enable_categorical=True, max_cat_to_onehot=1,  # Use the pandas categories directly.
        eval_metric="auc" if CFG.METRIC == "auc" else "logloss",
        early_stopping_rounds=CFG.EARLY_STOP, random_state=CFG.SEED, n_jobs=-1,
    )
    model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)], verbose=False)  # Stop using validation data.
    valid_probability = model.predict_proba(x_valid)[:, 1]  # Predictions use the best iteration automatically.
    test_probability = model.predict_proba(x_test)[:, 1]
    return valid_probability, test_probability


EMB_COLS = CFG.CAT_COLS + ["Age", "Number_of_Cars_Owned", "Charging_Stations_Near_Home", "Charging_Stations_Near_Work"]  # Learned vectors per value.
EMB_VOCAB = {name: sorted(set(train[name].astype(str)) | set(test[name].astype(str))) for name in EMB_COLS}  # Label-free.


class PeriodicEmbedding(nn.Module):
    """PLR embedding (Gorishniy et al., 2022): each number -> sin/cos at learned frequencies -> linear -> ReLU.

    Lets the network react to sharp changes in a number, the way a tree split does.
    """

    def __init__(self, n_num, k, dim, sigma):
        super().__init__()
        self.freq = nn.Parameter(torch.randn(n_num, k) * sigma)  # One set of frequencies per feature.
        self.weight = nn.Parameter(torch.randn(n_num, 2 * k, dim) / (2 * k) ** 0.5)
        self.bias = nn.Parameter(torch.zeros(n_num, dim))

    def forward(self, x):
        z = 2 * torch.pi * x[:, :, None] * self.freq[None]
        z = torch.cat([torch.sin(z), torch.cos(z)], -1)
        return torch.relu(torch.einsum("bnk,nkd->bnd", z, self.weight) + self.bias).flatten(1)


class TabularMLP(nn.Module):
    """Periodic embeddings for numbers + learned embeddings for categories, then a feed-forward network."""

    def __init__(self, n_num, cardinalities):
        super().__init__()
        self.plr = PeriodicEmbedding(n_num, CFG.PLR_K, CFG.PLR_DIM, CFG.PLR_SIGMA)
        self.embs = nn.ModuleList(nn.Embedding(size, min(8, size)) for size in cardinalities)
        width = n_num * CFG.PLR_DIM + sum(min(8, size) for size in cardinalities)
        layers = []
        for hidden in CFG.NN_HIDDEN:
            layers += [nn.Linear(width, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(CFG.NN_DROPOUT)]
            width = hidden
        self.net = nn.Sequential(*layers, nn.Linear(width, 1))

    def forward(self, num, cat):
        parts = [self.plr(num)] + [emb(cat[:, j]) for j, emb in enumerate(self.embs)]
        return self.net(torch.cat(parts, 1)).squeeze(1)  # One logit per row.


def fit_nn(x_train, y_train, x_valid, y_valid, x_test):
    """Fit one neural-network fold and return validation and test probabilities."""
    torch.manual_seed(CFG.SEED)  # Repeatable weights and batch order.
    num_cols = [name for name in x_train.columns if name not in CFG.CAT_COLS]  # Includes the in-fold TE_ columns.
    scaler = QuantileTransformer(n_quantiles=1000, output_distribution="normal", subsample=200_000, random_state=CFG.SEED)
    scaler.fit(x_train[num_cols].astype("float64"))  # Fit scaling on this fold's training rows only.

    def tensors(frame):
        num = torch.tensor(scaler.transform(frame[num_cols].astype("float64")), dtype=torch.float32)
        cat = torch.tensor(np.column_stack([pd.Categorical(frame[name].astype(str), categories=EMB_VOCAB[name]).codes
                                            for name in EMB_COLS]).astype("int64"))
        return num, cat

    def predict(model, num, cat):
        model.eval()
        with torch.no_grad():  # Score in large chunks to limit memory.
            return torch.cat([torch.sigmoid(model(num[i:i + 65536], cat[i:i + 65536]))
                              for i in range(0, len(num), 65536)]).numpy()

    num_tr, cat_tr = tensors(x_train)
    num_va, cat_va = tensors(x_valid)
    target = torch.tensor(y_train, dtype=torch.float32)
    model = TabularMLP(len(num_cols), [len(EMB_VOCAB[name]) for name in EMB_COLS])
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.NN_LR, weight_decay=1e-5)
    steps = (len(target) + CFG.NN_BATCH - 1) // CFG.NN_BATCH  # Gradient steps per epoch.
    schedule = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=CFG.NN_LR, total_steps=CFG.NN_EPOCHS * steps)
    loss_fn = nn.BCEWithLogitsLoss()
    best_score, best_state = -np.inf, None
    for epoch in range(CFG.NN_EPOCHS):
        model.train()
        order = torch.randperm(len(target))  # Shuffle rows every epoch.
        for step in range(steps):
            batch = order[step * CFG.NN_BATCH:(step + 1) * CFG.NN_BATCH]
            optimizer.zero_grad()
            loss_fn(model(num_tr[batch], cat_tr[batch]), target[batch]).backward()
            optimizer.step()
            schedule.step()
        epoch_score = (1 if CFG.METRIC == "auc" else -1) * score(y_valid, predict(model, num_va, cat_va))
        if epoch_score > best_score:  # Keep the best epoch, like early stopping in the tree models.
            best_score = epoch_score
            best_state = {key: value.clone() for key, value in model.state_dict().items()}
    model.load_state_dict(best_state)
    valid_probability = predict(model, num_va, cat_va)
    test_probability = predict(model, *tensors(x_test))
    return valid_probability, test_probability


def cross_validate(fit_model, name):
    """Train all folds, fill out-of-fold scores, and average test scores."""
    x = train[FEATURES]  # Use identical columns for both models.
    x_test = test[FEATURES]  # Select test columns in the same order.
    oof = np.zeros(len(train))  # Store one held-out prediction per training row.
    predicted_test = np.zeros(len(test))  # Accumulate the five test predictions.
    started = time.time()  # Record this model's start time.
    for number in range(CFG.N_FOLDS):  # Fit once for each validation fold.
        train_idx = np.flatnonzero(fold_id != number)  # Training rows for this fold.
        valid_idx = np.flatnonzero(fold_id == number)  # Held-out rows for this fold.
        x_train, x_valid, x_fold_test = x.iloc[train_idx], x.iloc[valid_idx], x_test  # This fold's tables.
        if CFG.TARGET_ENCODING:  # Encoders are refit inside every fold to avoid leakage.
            x_train, x_valid, x_fold_test = add_target_encoding(train_idx, valid_idx, x_train, x_valid, x_fold_test)
        valid_pred, test_pred = fit_model(x_train, y[train_idx], x_valid, y[valid_idx], x_fold_test)  # Fit and predict.
        oof[valid_idx] = valid_pred  # Put held-out scores in their original positions.
        predicted_test += test_pred / CFG.N_FOLDS  # Average test scores across folds.
        print(f"{name} fold {number + 1}: {CFG.METRIC}={score(y[valid_idx], valid_pred):.5f}")  # Report fold quality.
        gc.collect()  # Reclaim objects no longer held by the fold.
    run_minutes[name] = (time.time() - started) / 60  # Keep training time for the experiment log.
    print(f"{name} OOF {CFG.METRIC}={score(y, oof):.5f}; {run_minutes[name]:.1f} min")  # Report full CV quality.
    return oof, predicted_test  # Keep both arrays for blending.


run_minutes = {}  # Map each model name to its training time.
oof_predictions = {}  # Map each model name to its held-out scores.
test_predictions = {}  # Map each model name to its averaged test scores.
oof_predictions["LightGBM"], test_predictions["LightGBM"] = cross_validate(fit_lightgbm, "LightGBM")  # Run LightGBM.
if CFG.RUN_CATBOOST:  # Include CatBoost in the full baseline.
    oof_predictions["CatBoost"], test_predictions["CatBoost"] = cross_validate(fit_catboost, "CatBoost")
if CFG.RUN_XGBOOST:  # Exp07: a third model for the blend.
    oof_predictions["XGBoost"], test_predictions["XGBoost"] = cross_validate(fit_xgboost, "XGBoost")
if CFG.RUN_NN:  # Exp08: a different model family, so its mistakes differ from the trees'.
    oof_predictions["NeuralNet"], test_predictions["NeuralNet"] = cross_validate(fit_nn, "NeuralNet")

Loaded folds from folds_seed42.csv


LightGBM fold 1: auc=0.94506


LightGBM fold 2: auc=0.94574


LightGBM fold 3: auc=0.94692


LightGBM fold 4: auc=0.94645


...

NeuralNet fold 1: auc=0.94472


NeuralNet fold 2: auc=0.94531


NeuralNet fold 3: auc=0.94651


NeuralNet fold 4: auc=0.94602


NeuralNet fold 5: auc=0.94567
NeuralNet OOF auc=0.94555; 6.8 min


## 5. Blend and write the submission

Blend weights for all models are chosen together on out-of-fold predictions by hill climbing (ensemble selection). The final CSV follows the order and columns of `sample_submission.csv`.

In [5]:
results = {name: score(y, values) for name, values in oof_predictions.items()}  # Score each model once.
sign = 1 if CFG.METRIC == "auc" else -1  # Hill climbing maximises sign * score.


def hill_climb(oofs, steps=50):
    """Ensemble selection: repeatedly add (with replacement) the model that most improves the OOF score.

    Returns weights proportional to how often each model was picked, from the best step seen.
    """
    counts = dict.fromkeys(oofs, 0)
    first = max(oofs, key=lambda name: sign * score(y, oofs[name]))  # Start from the best single model.
    counts[first] = 1
    blend, best, best_counts = oofs[first].copy(), sign * score(y, oofs[first]), dict(counts)
    for step in range(1, steps):
        trials = {name: (blend * step + oofs[name]) / (step + 1) for name in oofs}  # Add one more vote to each model.
        trial_scores = {name: sign * score(y, values) for name, values in trials.items()}
        pick = max(trial_scores, key=trial_scores.get)
        counts[pick] += 1
        blend = trials[pick]
        if trial_scores[pick] > best:  # Keep the best combination found so far.
            best, best_counts = trial_scores[pick], dict(counts)
    total = sum(best_counts.values())
    return {name: count / total for name, count in best_counts.items()}


if len(oof_predictions) > 1:  # Choose all blend weights together.
    blend_weights = hill_climb(oof_predictions)
    final_oof = sum(weight * oof_predictions[name] for name, weight in blend_weights.items())  # Blend held-out scores.
    final_test = sum(weight * test_predictions[name] for name, weight in blend_weights.items())  # Blend test scores.
    results["Blend"] = score(y, final_oof)  # Report the blend.
else:  # A single model needs no blend search.
    blend_weights = {name: 1.0 for name in oof_predictions}
    final_oof = next(iter(oof_predictions.values()))
    final_test = next(iter(test_predictions.values()))
blend_note = ", ".join(f"{name} {weight:.2f}" for name, weight in blend_weights.items())  # Human-readable weights.
print("Blend weights:", blend_note)

print(pd.Series(results).round(5).to_string())  # Show the comparable CV scores.
prediction_column = sample.columns[1]  # Read the competition's probability column name.
lookup = pd.Series(final_test, index=test[CFG.ID].to_numpy())  # Match probabilities to test IDs.
submission = sample[[CFG.ID]].copy()  # Preserve sample submission row order.
submission[prediction_column] = submission[CFG.ID].map(lookup)  # Fill predictions by ID.
assert submission[prediction_column].notna().all(), "Some test IDs lack predictions."  # Check mapping completeness.
assert submission[prediction_column].between(0, 1).all(), "Predictions must be probabilities."  # Check output range.
assert list(submission.columns) == list(sample.columns), "Submission columns changed."  # Check required layout.
output_path = OUT_DIR / "submission.csv"  # Name the Kaggle upload artifact.
submission.to_csv(output_path, index=False)  # Write only the requested columns.
submission.to_csv(OUT_DIR / f"submission_{CFG.EXPERIMENT}.csv", index=False)  # Keep a copy per experiment.
print(f"Saved {output_path} and submission_{CFG.EXPERIMENT}.csv with {len(submission):,} rows")  # Confirm the output files.
submission.head()  # Preview the file in the notebook.

Blend weights: LightGBM 0.36, CatBoost 0.44, XGBoost 0.08, NeuralNet 0.13
LightGBM     0.94606
CatBoost     0.94606
XGBoost      0.94590
NeuralNet    0.94555
Blend        0.94616


Saved /Users/rodan/Documents/kaggle-september/submission.csv and submission_exp10_lgbm_params.csv with 286,571 rows


,id,Will_Buy_EV
0,668665,0.031934
1,668666,0.015312
2,668667,0.003773
3,668668,0.001789
4,668669,0.027319


## 6. Record the experiment

This cell appends one row per model to `experiments.csv` and saves the out-of-fold and test predictions in `preds/` for later blending. `delta_vs_baseline` and `folds_improved` compare this run with the same model in `CFG.BASELINE` on the same folds. Treat a gain as real only if it is at least +0.0002 and improves at least 4 of the 5 folds. After uploading the submission, write the public leaderboard score into the `public_lb` column of the `Blend` row.

In [6]:
def fold_scores(oof):
    """Score each fixed fold separately."""
    return [score(y[fold_id == k], oof[fold_id == k]) for k in range(CFG.N_FOLDS)]


def record_experiment():
    """Append this run to experiments.csv and save its predictions in preds/."""
    log_path = OUT_DIR / "experiments.csv"  # One row per (experiment, model).
    pred_dir = OUT_DIR / "preds"  # Saved predictions let later blends skip retraining.
    pred_dir.mkdir(exist_ok=True)
    suffix = "-fast" if CFG.FAST else ""  # FAST rows only compare with other FAST rows.
    runs = {**oof_predictions, "Blend": final_oof} if "Blend" in results else dict(oof_predictions)  # Models to log.
    tests = {**test_predictions, "Blend": final_test}  # Matching test predictions.
    log = pd.read_csv(log_path) if log_path.is_file() else pd.DataFrame()  # Previous experiments.
    rows = []
    for name, oof in runs.items():
        model = name + suffix
        folds = fold_scores(oof)
        base = log[(log.experiment == CFG.BASELINE) & (log.model == model)] if len(log) else log  # Same model in the baseline.
        delta, improved = np.nan, np.nan
        if len(base) and CFG.EXPERIMENT != CFG.BASELINE:
            delta = score(y, oof) - base.oof_auc.iloc[0]
            base_folds = [float(v) for v in base.fold_aucs.iloc[0].split()]
            improved = sum(f > b for f, b in zip(folds, base_folds))
        previous = log[(log.experiment == CFG.EXPERIMENT) & (log.model == model)] if len(log) else log  # Keep a recorded LB score on reruns.
        rows.append({
            "experiment": CFG.EXPERIMENT, "model": model, "n_features": len(FEATURES) + (len(TRAIN_KEYS) if CFG.TARGET_ENCODING else 0),
            "fold_aucs": " ".join(f"{f:.5f}" for f in folds), "oof_auc": round(score(y, oof), 5),
            "delta_vs_baseline": round(delta, 5), "folds_improved": improved,
            "minutes": round(run_minutes.get(name, 0.0), 1), "fold_seed": CFG.SEED,
            "notes": blend_note if name == "Blend" else "",
            "public_lb": previous.public_lb.iloc[0] if len(previous) else np.nan,
            "logged_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
        })
        np.save(pred_dir / f"{CFG.EXPERIMENT}_{model}_oof.npy", oof)
        np.save(pred_dir / f"{CFG.EXPERIMENT}_{model}_test.npy", tests[name])
    if len(log):  # Replace earlier rows from a rerun of the same experiment.
        log = log[~((log.experiment == CFG.EXPERIMENT) & log.model.isin([r["model"] for r in rows]))]
    log = pd.concat([log, pd.DataFrame(rows)], ignore_index=True)
    log.to_csv(log_path, index=False)
    return log


if CFG.DEBUG:  # Debug runs use a sample and would pollute the log.
    print("DEBUG run: experiment not recorded.")
else:
    experiment_log = record_experiment()
    shown = experiment_log[experiment_log.experiment.isin([CFG.BASELINE, CFG.EXPERIMENT])]
    print(shown.drop(columns=["logged_at"]).to_string(index=False))


         experiment     model  n_features                               fold_aucs  oof_auc  delta_vs_baseline  folds_improved  minutes  fold_seed                                                      notes  public_lb
exp09_wide_encoding  LightGBM          65 0.94495 0.94558 0.94662 0.94628 0.94589  0.94585            0.00038             5.0      1.8         42                                                        NaN        NaN
exp09_wide_encoding  CatBoost          65 0.94521 0.94577 0.94688 0.94637 0.94612  0.94606            0.00038             5.0      5.1         42                                                        NaN        NaN
exp09_wide_encoding   XGBoost          65 0.94499 0.94563 0.94670 0.94621 0.94599  0.94590            0.00038             5.0      1.5         42                                                        NaN        NaN
exp09_wide_encoding NeuralNet          65 0.94472 0.94531 0.94651 0.94602 0.94567  0.94555            0.00040             5.0      7.5  

## 7. Diagnostics and interpretation

The four charts show purchase rate by category, numeric distributions, separation of held-out probabilities, and LightGBM feature importance. They help inspect the model; they do not affect `submission.csv`.

In [7]:
pio.templates.default = "plotly_dark"  # Give all four charts a consistent dark theme.
positive_rate = y.mean()  # Use the training base rate as a reference line.
raw_features = [name for name in pd.read_csv(DATA_DIR / "test.csv", nrows=0).columns if name != CFG.ID]  # Exclude engineered columns.
raw_numeric = [name for name in raw_features if name not in CFG.CAT_COLS]  # Choose original numeric inputs.

# Chart 1: compare category purchase rates with the global purchase rate.
category_rows = (len(CFG.CAT_COLS) + 1) // 2  # Fit two category charts per row.
fig = make_subplots(rows=category_rows, cols=2, subplot_titles=CFG.CAT_COLS)  # Create the grid.
for index, name in enumerate(CFG.CAT_COLS):  # Draw one panel per category.
    row, col = divmod(index, 2)  # Convert the index to a grid position.
    grouped = train.groupby(name, observed=True)[CFG.TARGET].agg(["mean", "size"]).sort_values("mean")  # Measure purchase rate and count.
    fig.add_bar(x=grouped["mean"], y=grouped.index.astype(str), orientation="h", customdata=grouped["size"],
                marker_color="#3987e5", hovertemplate="%{y}: %{x:.1%}<br>n=%{customdata:,}<extra></extra>",
                showlegend=False, row=row + 1, col=col + 1)  # Show rates with sample sizes.
    fig.add_vline(x=positive_rate, line_color="#aaaaaa", row=row + 1, col=col + 1)  # Mark the base rate.
fig.update_xaxes(tickformat=".0%")  # Display probabilities as percentages.
fig.update_layout(title="Purchase rate by category", height=320 * category_rows)  # Fit all panels.
fig.show()  # Render the first chart.

# Chart 2: compare each raw numeric feature's distribution by target class.
numeric_rows = (len(raw_numeric) + 1) // 2  # Fit two numeric features per row.
fig = make_subplots(rows=numeric_rows, cols=2, subplot_titles=raw_numeric)  # Create the grid.
for index, name in enumerate(raw_numeric):  # Draw one panel per numeric input.
    row, col = divmod(index, 2)  # Convert the index to a grid position.
    for label, color in ((0, "#3987e5"), (1, "#d95926")):  # Plot both target classes.
        values = train.loc[train[CFG.TARGET] == label, name].dropna()  # Select that class's values.
        hist, edges = np.histogram(values, bins=40, density=True)  # Estimate its density.
        centers = (edges[:-1] + edges[1:]) / 2  # Center each histogram bin.
        fig.add_scatter(x=centers, y=hist, mode="lines", line_color=color,
                        name=f"target={label}", legendgroup=str(label), showlegend=index == 0,
                        row=row + 1, col=col + 1)  # Overlay the class densities.
fig.update_layout(title="Numeric feature distributions by class", height=270 * numeric_rows)  # Fit all panels.
fig.show()  # Render the second chart.

# Chart 3: inspect how well the out-of-fold probabilities separate the classes.
fig = go.Figure()  # Start a standalone probability chart.
for label, color in ((0, "#3987e5"), (1, "#d95926")):  # Draw each target class.
    values = final_oof[y == label]  # Select held-out scores for the class.
    fig.add_histogram(x=values, histnorm="probability", nbinsx=50, opacity=0.55,
                      marker_color=color, name=f"target={label}")  # Compare probability mass.
fig.update_layout(barmode="overlay", title=f"Out-of-fold prediction separation; {CFG.METRIC}={score(y, final_oof):.5f}",
                  xaxis_title="Predicted probability", yaxis_title="Share of class")  # Explain axes.
fig.show()  # Render the third chart.

# Chart 4: rank LightGBM features by the gain they contributed in the last fold.
last_model = fit_lightgbm.last_model  # Reuse the model kept from cross-validation.
gain = last_model.booster_.feature_importance("gain")  # Read total split gain per feature.
importance = pd.Series(gain, index=last_model.feature_name_).sort_values()  # Match gains to feature names.
importance = importance / importance.sum()  # Convert raw gain to a share of total gain.
fig = go.Figure(go.Bar(x=importance.values, y=importance.index, orientation="h", marker_color="#3987e5"))  # Draw ranked bars.
fig.update_layout(title="LightGBM feature importance (last fold)", xaxis_tickformat=".0%",
                  xaxis_title="Share of total gain", height=max(450, 28 * len(importance)))  # Format the chart.
fig.show()  # Render the fourth chart.